# Criação de origem_destino_customedio.csv

Esse notebook é utilizado para carregar os dados brutos e criar a tabela dados_processados/origem_destino_customedio.csv. Essa tabela nos informa o preço médio cobrado, por cada compania aérea, para cada Origem -> Destino.

In [4]:
import glob
import pandas as pd 
import numpy as np  

A base de dados nos dá informações sobre quanto foi pago em tarifas aéreas no ano de 2026. A base de dados está imcompleta, mas já tem dados até setembro. Esse código lê as tabelas sobre tarifas aéreas de cada mês disponível em 2026:

In [5]:
# sintaxe de generator function. funciona na base da fé!
df = pd.concat(
    (pd.read_csv(f, sep=';', encoding='latin-1') for f in glob.glob('dados_brutos/*.csv')),
    ignore_index=True,
)
df.sample(5)

ValueError: No objects to concatenate

In [ ]:
df.columns

df = df.rename(columns={'nr_ano_referencia': 'ano', 'nr_mes_referencia': 'mes', 'sg_empresa_icao': 'empresa',
                   'sg_icao_origem': 'origem', 'sg_icao_destino': 'destino', 'nr_tarifa': "valor_tarifa"})

Ajustando o tipo da coluna 'valor_tarifa' de str para float

In [ ]:
display(df.dtypes)
# O valor da tarifa está como str!
print(df.columns)

def valor_tarifa_para_float(valor: str) -> float:
    valor = valor.replace(",", ".")
    valor = float(valor)
    valor = np.float16(valor)
    return valor

df['valor_tarifa'] = df['valor_tarifa'].apply(valor_tarifa_para_float)

df.dtypes

# Eba!!!

ano             int64
mes             int64
empresa           str
origem            str
destino           str
valor_tarifa      str
nr_assentos     int64
dtype: object

Index(['ano', 'mes', 'empresa', 'origem', 'destino', 'valor_tarifa',
       'nr_assentos'],
      dtype='str')


ano               int64
mes               int64
empresa             str
origem              str
destino             str
valor_tarifa    float16
nr_assentos       int64
dtype: object

Verificando dados faltantes (não há células em branco!)

In [ ]:
print(f"Dados faltantes totais:\n{df.isna().sum()}")
# AEEE!!!

Dados faltantes totais:
ano             0
mes             0
empresa         0
origem          0
destino         0
valor_tarifa    0
nr_assentos     0
dtype: int64


In [ ]:
df["custo_por_assento"] = df["valor_tarifa"] / df["nr_assentos"]
df.sample(5)

c:\Users\Diogo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pandas\io\formats\format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,ano,mes,empresa,origem,destino,valor_tarifa,nr_assentos,custo_por_assento
6926440,2026,9,AZU,SBSI,SBVT,1985.00,1,1985.000000
6783315,2026,9,AZU,SBRF,SBSV,274.25,3,91.416667
2442737,2026,4,AZU,SBJP,SBPA,435.00,4,108.750000
715679,2026,1,TAM,SBRF,SBSP,1614.00,2,807.000000
6165869,2026,8,GLO,SBRJ,SBSP,408.25,1,408.250000


In [ ]:
# reset_index faz ser retornado um df
origem_destino_customedio = df.groupby(["empresa", "origem", "destino"])["custo_por_assento"].mean().reset_index()
origem_destino_customedio

,origem,destino,custo_por_assento
0,SBAC,SBCF,1166.250000
1,SBAC,SBGL,873.000000
2,SBAC,SBGR,593.500000
3,SBAC,SBKP,248.375000
4,SBAC,SBVT,460.583333
...,...,...,...
8725,SWPI,SBTT,1564.500000
8726,SWPI,SBUL,887.000000
8727,SWPI,SNBR,3640.000000
8728,SWYN,SBEG,347.963803


In [ ]:
origem_destino_customedio.to_csv(path_or_buf="dados_processados/origem_destino_customedio.csv", sep=";")